In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv(r".\Dataset\cmi_internet.csv")
df.shape

In [ ]:
df.isna().sum()

In [ ]:
existing_counts = df.notna().sum()
print(existing_counts)

In [ ]:
mask = df.notna().sum() > 4900
filtered_df = df.loc[:, mask]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

def find_and_apply_best_imputation_final(df, k_folds=5):
    results = {}
    # 1. Create a copy of the dataframe to store our actual imputations
    df_imputed = df.copy() 
    
    numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    for target_col in numeric_cols:
        # Skip columns that don't actually have missing data
        if df_imputed[target_col].isna().sum() == 0:
            continue
            
        # Separate rows with data (for training) and rows without (for predicting)
        valid_mask = df_imputed[target_col].notna()
        missing_mask = df_imputed[target_col].isna()
        
        valid_data = df_imputed[valid_mask]
        
        # Campionamento (Using min() just in case a column has less than 4900 valid rows)
        n_sample = min(4900, len(valid_data))
        sample_data = valid_data.sample(n=n_sample, random_state=42)
        
        y_train = sample_data[target_col]
        X_train_raw = sample_data.drop(columns=[target_col])
        X_missing_raw = df_imputed[missing_mask].drop(columns=[target_col])
        
        # 2. Combine before get_dummies to ensure training and missing sets have identical columns
        X_combined = pd.concat([X_train_raw, X_missing_raw])
        X_combined_dummies = pd.get_dummies(X_combined, drop_first=True)
        
        # Split them back apart
        X_train = X_combined_dummies.iloc[:len(X_train_raw)]
        X_missing = X_combined_dummies.iloc[len(X_train_raw):]
        
        # Imputiamo eventuali NaN rimasti nelle feature predittive con la mediana
        imputer = SimpleImputer(strategy='median')
        X_train_imputed = imputer.fit_transform(X_train)
        X_missing_imputed = imputer.transform(X_missing) # Use transform() here, not fit_transform()
        
        # 3. Calcolo di K per il KNN
        k_neighbors = int(np.sqrt(len(X_train_imputed)))
        
        knn_pipeline = Pipeline([
            ('scaler', MinMaxScaler()),
            ('knn', KNeighborsRegressor(n_neighbors=k_neighbors))
        ])
        
        models = {
            'Linear Regression': LinearRegression(),
            'Decision Tree': DecisionTreeRegressor(random_state=42),
            f'KNN (k={k_neighbors})': knn_pipeline
        }
        
        best_r2 = -float('inf')
        best_method = ""
        best_model_instance = None
        
        # 4. Valutazione con K-Fold
        for name, model in models.items():
            scores = cross_val_score(model, X_train_imputed, y_train, cv=kf, scoring='r2')
            avg_r2 = np.mean(scores)
            
            if avg_r2 > best_r2:
                best_r2 = avg_r2
                best_method = name
                best_model_instance = model
        
        results[target_col] = {
            'Miglior Metodo': best_method, 
            'R2 Score': round(best_r2, 4)
        }
        
        # 5. NEW: Apply the changes if R2 is greater than 0
        if best_r2 > 0:
            # Fit the winning model on the complete training set
            best_model_instance.fit(X_train_imputed, y_train)
            
            # Predict the missing values
            predictions = best_model_instance.predict(X_missing_imputed)
            
            # Inject the predictions back into the copied dataframe
            df_imputed.loc[missing_mask, target_col] = predictions
            
            results[target_col]['Status'] = 'Imputed'
        else:
            results[target_col]['Status'] = 'Skipped (R2 <= 0)'
        
    return df_imputed, pd.DataFrame(results).T

# --- Utilizzo ---
# The function now returns TWO items: the newly filled dataframe, and the report.
filled_df, report = find_and_apply_best_imputation_final(filtered_df)

print(report)
# You can now use `filled_df` for your downstream tasks

In [ ]:
non_imputed_cols = ['CGAS-CGAS_Score', 'BIA-BIA_BMC', 'BIA-BIA_ECW', 'BIA-BIA_FFM', 'BIA-BIA_Fat', 'BIA-BIA_LDM', 'BIA-BIA_SMM', 'BIA-BIA_TBW']

In [ ]:
filled_df['CGAS-CGAS_Score'].describe()

In [ ]:
(filled_df['CGAS-CGAS_Score'] > 500).sum()

In [ ]:
for col in non_imputed_cols:
    plt.figure()
    sns.histplot(filled_df[col])
    plt.show()

In [ ]:
df[]